In [109]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import os

In [110]:
repub_df_1 = pd.read_csv('{}/results/human_annotations/NYTimes Data Collection - Republican_May 27, 2024_15.36.csv'.format(os.getcwd()))
repub_df_1 = repub_df_1.drop(index=[0, 1])

dem_df_1 = pd.read_csv('{}/results/human_annotations/NYTimes Data Collection - Democrat_May 27, 2024_15.58.csv'.format(os.getcwd()))
dem_df_1 = dem_df_1.drop(index=[0, 1])

repub_df_2 = pd.read_csv('{}/results/human_annotations/NYTimes Data Collection - Republican - V2_May 27, 2024_22.33.csv'.format(os.getcwd()))
repub_df_2 = repub_df_2.drop(index=[0, 1])

dem_df_2 = pd.read_csv('{}/results/human_annotations/NYTimes Data Collection - Democrat - V2_May 27, 2024_22.33.csv'.format(os.getcwd()))
dem_df_2 = dem_df_2.drop(index=[0, 1])


In [111]:
dem_df = pd.concat([dem_df_1, dem_df_2], ignore_index=True, sort=False)
repub_df = pd.concat([repub_df_1, repub_df_2], ignore_index=True, sort=False)

In [112]:
combined_df = pd.concat([dem_df, repub_df],  ignore_index=True, sort=False)
col_to_keep = ['QDem_Gend', 'QDem_Race', 'Q_DemRepub', 'QDem_Income', 'Q84', 'QDem_Age'] # demographics
combined_df = combined_df[col_to_keep]
combined_df.to_csv('{}/results/human_annotations/nytimes_annotator_demographics.csv'.format(os.getcwd()))

In [113]:
data = pd.read_csv('{}/results/human_annotations/NYTIMES_BOOK_DATASET.csv'.format(os.getcwd()))


In [114]:
qualtricsID_to_title, title_to_qualtricsID = {}, {}
col_to_keep = ['Q84', 'QDem_Age', 'QDem_Gend', 'QDem_Race', 'Q_DemRepub', 'QDem_Income'] # demographics
booktitles = np.array(data['Title and Author'])
genres = np.array(data['Genre'])
summaries = np.array(data['Summary'])
for i, qualtricsID in enumerate([1] + list(np.arange(307, 333))+ list(np.arange(334, 385))+ list(np.arange(386, 408)) + [100] + list(np.arange(408,542))):
  col_to_keep.append(str(qualtricsID)+'_Q138')
  if str(qualtricsID)+'_Q138' not in repub_df.columns: print(str(qualtricsID)+'_Q138')
  qualtricsID_to_title[qualtricsID] = booktitles[i]
  title_to_qualtricsID[booktitles[i]] = qualtricsID

In [115]:
repub_df = repub_df[col_to_keep]
dem_df = dem_df[col_to_keep]

dem_df['QDem_Age'] = dem_df['QDem_Age'].astype(int)
repub_df['QDem_Age'] = repub_df['QDem_Age'].astype(int)

In [63]:
# male female
male_df = pd.concat([repub_df[repub_df['QDem_Gend']=='Male'], dem_df[dem_df['QDem_Gend']=='Male']], ignore_index=True, sort=False)
female_df = pd.concat([repub_df[repub_df['QDem_Gend']=='Female'], dem_df[dem_df['QDem_Gend']=='Female']], ignore_index=True, sort=False)

repub_df = repub_df[repub_df['Q_DemRepub']=='Republican']
dem_df = dem_df[dem_df['Q_DemRepub']=='Democrat']



In [64]:
len(male_df), len(female_df), len(repub_df), len(dem_df) 

(131, 206, 165, 172)

In [ ]:
dem_values, repub_values, male_values, female_values = [], [], [], []
dem_data, repub_data, male_data, female_data = [], [], [], []
dem_id_data, repub_id_data, male_id_data, female_id_data = [], [], [], []
for i, qualtricsID in enumerate([1] + list(np.arange(307, 333))+ list(np.arange(334, 385))+ list(np.arange(386, 408)) + [100] + list(np.arange(408,542))):
    print(str(qualtricsID)+'_Q138')
    non_nan_values = np.array(dem_df[str(qualtricsID)+'_Q138'].dropna())
    non_nan_prolific_ids = dem_df['Q84'][~pd.isna(dem_df[str(qualtricsID)+'_Q138'])]
    dem_data.append(non_nan_values)
    dem_id_data.append(dict(zip(non_nan_prolific_ids, non_nan_values)))
    dem_values.append(len(non_nan_values))

    non_nan_values = np.array(repub_df[str(qualtricsID)+'_Q138'].dropna())
    non_nan_prolific_ids = repub_df['Q84'][~pd.isna(repub_df[str(qualtricsID)+'_Q138'])]
    repub_data.append(non_nan_values)
    repub_id_data.append(dict(zip(non_nan_prolific_ids, non_nan_values)))
    repub_values.append(len(non_nan_values))

    non_nan_values = np.array(male_df[str(qualtricsID)+'_Q138'].dropna())
    non_nan_prolific_ids = male_df['Q84'][~pd.isna(male_df[str(qualtricsID)+'_Q138'])]
    male_id_data.append(dict(zip(non_nan_prolific_ids, non_nan_values)))
    male_data.append(non_nan_values)
    male_values.append(len(non_nan_values))

    non_nan_values = np.array(female_df[str(qualtricsID)+'_Q138'].dropna())
    non_nan_prolific_ids = female_df['Q84'][~pd.isna(female_df[str(qualtricsID)+'_Q138'])]
    female_id_data.append(dict(zip(non_nan_prolific_ids, non_nan_values)))
    female_data.append(non_nan_values)
    female_values.append(len(non_nan_values))

dem_data = np.array(dem_data, dtype=object)
repub_data = np.array(repub_data, dtype=object)

male_data = np.array(male_data, dtype=object)
female_data = np.array(female_data, dtype=object)

In [67]:
# convert all the labels to numbers
labels_to_num = {'1: Very unlikely':1 , '2: Somewhat unlikely':2, '3: Somewhat likely':3, '4: Very likely': 4}
def get_nums(x):
    return labels_to_num[x]

vectorized_function = np.vectorize(get_nums)

# Apply the vectorized function to the array
dem_data_nums = []
for i in range(len(dem_data)):
    dem_data_nums.append(vectorized_function(dem_data[i]))

repub_data_nums = []
for i in range(len(repub_data)):
    repub_data_nums.append(vectorized_function(repub_data[i]))

male_data_nums = []
for i in range(len(male_data)):
    male_data_nums.append(vectorized_function(male_data[i]))

female_data_nums = []
for i in range(len(female_data)):
    female_data_nums.append(vectorized_function(female_data[i]))

In [70]:
num_to_label = {1: '1: Very unlikely', 2: '2: Somewhat unlikely', 3: '3: Somewhat likely', 4: '4: Very likely'}


count = 0 
all = []
proportions_data_annotator_id = defaultdict(dict)
for i, book_title in enumerate(booktitles): 
    book_title=str(book_title)
    proportions_data_annotator_id[book_title]['MC_options']=list(['1: Very unlikely', '2: Somewhat unlikely', '3: Somewhat likely', '4: Very likely'])
    proportions_data_annotator_id[book_title]['genre']= genres[i]
    proportions_data_annotator_id[book_title]['summary']= summaries[i]
    
    proportions_data_annotator_id[book_title]['Democrat'] = dem_id_data[i]
    proportions_data_annotator_id[book_title]['Republican'] = repub_id_data[i]
    proportions_data_annotator_id[book_title]['Male'] = male_id_data[i]
    proportions_data_annotator_id[book_title]['Female'] = female_id_data[i]

    all.append(len(dem_id_data[i].keys()))
    all.append(len(repub_id_data[i].keys()))
    all.append(len(male_id_data[i].keys()))
    all.append(len(female_id_data[i].keys()))
    count += len(dem_id_data[i].keys())+len(repub_id_data[i].keys())+len(male_id_data[i].keys())+len(female_id_data[i].keys())

print(count)


17570


In [ ]:

def count_keys(keys, lst):
    # Initialize a dictionary with all keys set to 0
    key_counts = {key: 0 for key in keys}
    
    # Count occurrences of each key in the list
    for item in lst:
        if item in key_counts:
            key_counts[item] += 1
    
    return key_counts

mc_options = [1, 2, 3, 4]

def get_label_to_counts(my_list):
    print(my_list)
    label_to_count = count_keys(mc_options, my_list)
    print(label_to_count)
    # labels, counts = np.unique(my_list, return_counts=True)
    # label_to_count = dict(zip(labels,counts))
    converted_data = {num_to_label[k]: int(v) for k, v in label_to_count.items()}
    return converted_data


proportions_data = defaultdict(dict)
for i, book_title in enumerate(booktitles): 
    book_title=str(book_title)
    proportions_data[book_title]['MC_options']=list(['1: Very unlikely', '2: Somewhat unlikely', '3: Somewhat likely', '4: Very likely'])
    proportions_data[book_title]['genre']= genres[i]
    proportions_data[book_title]['summary']= summaries[i]
    
    proportions_data[book_title]['Democrat'] = get_label_to_counts(dem_data_nums[i])
    proportions_data[book_title]['Republican'] = get_label_to_counts(repub_data_nums[i])
    proportions_data[book_title]['Male'] = get_label_to_counts(male_data_nums[i])
    proportions_data[book_title]['Female'] = get_label_to_counts(female_data_nums[i])


In [73]:
# Specify the filename
filename = '{}/NYTIMES/NYTIMES_proportions.json'.format(os.getcwd())

# Open the file in write mode and use json.dump to write the dictionary to the file
with open(filename, 'w') as file:
    json.dump(proportions_data, file, indent=4)

#### Finding top 10 similar books 

In [20]:
from fuzzywuzzy import fuzz
from simcse import SimCSE
from sklearn.metrics.pairwise import cosine_similarity

model = SimCSE("princeton-nlp/sup-simcse-bert-base-uncased")

In [ ]:
#### TOP 10 similar questions
import json
data_filename = '{}/nytimes/individual_annotations/NYTIMES_proportions.json'.format(os.getcwd())

with open(data_filename, 'r') as json_file:
    data = json.load(json_file)

question_similarity = {}
# map question IDs to embeddings 
question_ID_to_embedding = {}

# go through all the questions in all the waves and calc text embeddings


for q_ID in list(data.keys()):
    prompt=''
    prompt+= "\nBook Title: " + q_ID 
    prompt+= "\nBook Genre: " + data[q_ID]['genre'] 
    prompt+= "\nBook Summary: " + data[q_ID]['summary'] 
    embeddings = np.array(model.encode(prompt).tolist()).reshape(-1, 1)
    question_ID_to_embedding[q_ID] = embeddings


100%|██████████| 1/1 [00:00<00:00,  1.70it/s]


In [ ]:
for question_ID in data.keys():
    question_IDs, question_IDs_similarity = [], []
    
    for question_ID_other in data.keys():
        if question_ID_other!=question_ID: 
            question_IDs.append(question_ID_other)

            # find similar keys by comparing question ID
            a, b = question_ID_to_embedding[question_ID], question_ID_to_embedding[question_ID_other] 
            norm = np.linalg.norm(a - b) # smaller is more similar 
            question_IDs_similarity.append(norm)
    
    sorted_pairs = sorted(zip(question_IDs_similarity, question_IDs))
    sorted_listA = [pair[1] for pair in sorted_pairs]
    question_similarity[question_ID] = sorted_listA[:10]

In [15]:
# save data 
with open('{}/nytimes/question_similarity_top10.json'.format(os.getcwd()), "w") as json_file:
    json.dump(question_similarity, json_file)
